## Preparation

In [1]:
# --- SETUP & INSTALLATIONS ---
# Ensure dependencies are present
# !pip install openpyxl fingertips_py

In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os
import requests
import time
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
# SETUP
RAW_DIR = r"./data/raw"
PROCESSED_DIR = r"./data/processed"

# Files have been downloaded locally due to blocks on direct downloads in some of the data 
# sources. Ensure these files are placed in the 'data/raw' directory. Define the specific 
# filenames as they exist in the raw folder.

# MSOA lookup file (contains the mapping of MSOAs to LSOA and LADs)
# Download from: https://www.data.gov.uk/dataset/c2235117-cbfd-480d-8fc7-b564bd0f4d58/output-area-2021-to-lsoa-to-msoa-to-lep-to-lad-december-2022-best-fit-lookup-in-en-v2
LOOKUP_FILE = "OA21_LAD22_LSOA21_MSOA21_LEP22_EN_LU_V2_6716459600479702985.csv"

# Access to green space data (from OS Open Greenspace)
# Download from: https://www.ons.gov.uk/economy/environmentalaccounts/datasets/accesstogardensandpublicgreenspaceingreatbritain
GREEN_FILE = "ospublicgreenspacereferencetables.xlsx"

# Indicies of Multiple Deprivation (IMD) data
# Download from: https://www.gov.uk/government/statistics/english-indices-of-deprivation-2025
IMD_FILE = "File_7_IoD2025_All_Ranks_Scores_Deciles_Population_Denominators.csv"

# TfL Public Transport Accessibility Levels (PTAL) data
# Download from: https://gis-tfl.opendata.arcgis.com/datasets/20cf34d949404dd9bffe48c0b5e31b8b_31/explore
PTAL_FILE = "MSOA_aggregated_PTAL_stats_2023.csv"

if not os.path.exists(PROCESSED_DIR):
    os.makedirs(PROCESSED_DIR)

## Prepare London MSOA lookups for LSOA and LAD

In [4]:
# Prepare the Master London List using the lookup
print("Loading London Lookup...")
df_lookup = pd.read_csv(os.path.join(RAW_DIR, LOOKUP_FILE), low_memory=False)

# Filter for London (LAD codes starting with E09)
london_master = df_lookup[df_lookup['LAD22CD'].str.startswith('E09', na=False)].copy()

# Create MSOA-level master list for London
london_msoas = london_master[['LSOA21CD', 'LSOA21NM','LAD22CD', 'LAD22NM', 'MSOA21CD', 'MSOA21NM']].drop_duplicates()
london_msoas.columns = ['lsoa_code', 'lsoa_name', 'lad_code', 'lad_name', 'msoa_code', 'msoa_name']

print("... London Master lookup created")

Loading London Lookup...
... London Master lookup created


## Obesity Data

In [5]:
# OBESITY DATA (Y VARIABLE)
print("Processing Obesity Data...")
TARGET_PERIOD = "2022/23 - 24/25" # latest available period for obesity data
OBESITY_URL = "https://fingertips.phe.org.uk/api/all_data/csv/by_indicator_id?indicator_ids=93107&child_area_type_id=3&parent_area_type_id=15"

# Load obesity data and filter for the target period
df_england = pd.read_csv(OBESITY_URL, low_memory=False)
df_period = df_england[df_england['Time period'] == TARGET_PERIOD].copy()
df_period = df_period.rename(columns={'Area Code': 'msoa_code', 'Value': 'child_obesity_rate'})
df_period['child_obesity_rate'] = pd.to_numeric(df_period['child_obesity_rate'], errors='coerce')

df_obesity = df_period[['msoa_code', 'child_obesity_rate']].drop_duplicates(subset=['msoa_code']).copy()

# FILTER FOR LONDON AND GUARANTEE UNIQUENESS
df_obesity = df_period[df_period['msoa_code'].isin(london_msoas['msoa_code'])].copy()

# Keep necessary columns and drop duplicates to ensure 1 row per MSOA
df_obesity = df_obesity[['msoa_code', 'child_obesity_rate']].drop_duplicates(subset=['msoa_code'])

print(f"... Obesity data filtered for London. Unique MSOAs: {len(df_obesity)}")

Processing Obesity Data...
... Obesity data filtered for London. Unique MSOAs: 1002


## Access to Greenspace

In [6]:
# ACCESS TO GREENSPACE (X VARIABLE)
print("Processing Greenspace Data...")
df_green_raw = pd.read_excel(os.path.join(RAW_DIR, GREEN_FILE), sheet_name='LSOA Parks and Playing Fields')

# Define the columns and the aggregation method (Mean)
green_metrics = {
    #'Average distance to nearest Park, Public Garden, or Playing Field (m)': 'mean',
    #'Average size of nearest Park, Public Garden, or Playing Field (m2)': 'mean',
    'Average combined size of  Parks, Public Gardens, or Playing Fields within 1,000 m radius (m2)': 'mean'
}

# Aggregate from LSOA-level rows to MSOA-level averages
df_green = df_green_raw.groupby('MSOA code').agg(green_metrics).reset_index()
df_green = df_green.rename(columns={'MSOA code': 'msoa_code'})

# Filter to London MSOAs
df_green = df_green[df_green['msoa_code'].isin(london_msoas['msoa_code'])].copy()

# 2.3 Final column naming for the model
df_green.columns = [
    'msoa_code', 
    #'avg_dist_green_m', 
    #'avg_size_green_m2', 
    'avg_size_green_1km_m2'
]

print(f"... Greenspace data filtered for London. Unique MSOAs: {len(df_green)}")

Processing Greenspace Data...
... Greenspace data filtered for London. Unique MSOAs: 963


## IMD

In [7]:
# --- 3. IMD SCORES (X VARIABLES) ---
print("Processing IMD Data...")
df_imd_raw = pd.read_csv(os.path.join(RAW_DIR, IMD_FILE))

imd_map = {
    'Employment Score (rate)': 'employment_score',
    'Education, Skills and Training Score': 'education_score',
    'Crime Score': 'crime_score',
    'Income Deprivation Affecting Children Index (IDACI) Score (rate)': 'idaci_score'
}

# Aggregate LSOA scores to MSOA mean using the London mapping
lsoa_to_msoa = london_msoas[['lsoa_code', 'msoa_code']].drop_duplicates()
df_imd_london = pd.merge(lsoa_to_msoa, df_imd_raw, left_on='lsoa_code', right_on='LSOA code (2021)', how='inner')

df_imd = df_imd_london.groupby('msoa_code')[list(imd_map.keys())].mean().reset_index()
df_imd = df_imd.rename(columns=imd_map)

print(f"... IMD data processed for London. Unique MSOAs: {len(df_imd)}")

Processing IMD Data...
... IMD data processed for London. Unique MSOAs: 1002


## PTAL

In [8]:
# PUBLIC TRANSPORT ACCESSIBILITY LEVELS (PTAL) DATA (NEW X VARIABLE)
print("Processing PTAL Data...")

df_ptal_raw = pd.read_csv(os.path.join(RAW_DIR, PTAL_FILE))

# We ensure it's a copy and drop any duplicates to keep it 1-to-1 with MSOAs
df_ptal = df_ptal_raw[['MSOA21CD', 'mean_AI']].drop_duplicates(subset=['MSOA21CD']).copy()

# Rename for clarity in the final model (optional)
df_ptal = df_ptal.rename(columns={'MSOA21CD': 'msoa_code', 'mean_AI': 'ptal_accessibility_index'})

print(f"... PTAL data processed. Unique MSOAs: {len(df_ptal)}")

Processing PTAL Data...
... PTAL data processed. Unique MSOAs: 1002


## Takeaway Locations

Connecting to the Food Hygiene Rating Service's API of food premise locations to download the takeaway locations in London. This is then used to calculate takeaway density per MSOAs to investigate the relationship between fast food and childhood obesity.

### Part 1 - Download Food locations

This connects to the FHRS API and downloads the takeaways for London. This should only be run once to prevent overloading the API.

In [9]:
def get_london_fast_food_data():
    api_url = "https://api.ratings.food.gov.uk"
    headers = {'x-api-version': '2', 'accept': 'application/json'}
    
    # --- Step 1: Get London Authority IDs ---
    print("Fetching London Authority list...")
    auth_resp = requests.get(f"{api_url}/Authorities", headers=headers)
    auth_resp.raise_for_status()
    all_auths = auth_resp.json()['authorities']
    
    # Filter for London (Region ID 3 / RegionName 'London')
    london_boroughs = [
        {'id': a['LocalAuthorityId'], 'name': a['Name']} 
        for a in all_auths if a.get('RegionName') == 'London'
    ]
    print(f"Found {len(london_boroughs)} London boroughs.")

    all_establishments = []

    # --- Step 2: Loop through each borough and fetch takeaways ---
    for borough in london_boroughs:
        auth_id = borough['id']
        auth_name = borough['name']
        print(f"Processing {auth_name} (ID: {auth_id})...")
        
        page = 1
        while True:
            params = {
                'localAuthorityId': auth_id,
                'businessTypeId': 7844, # businessTypeId 7844 = Takeaway/sandwich shop
                'pageSize': 200,
                'pageNumber': page
            }
            
            response = requests.get(f"{api_url}/Establishments", headers=headers, params=params)
            data = response.json()
            establishments = data.get('establishments', [])
            
            if not establishments:
                break
                
            for est in establishments:
                # IMPORTANT: Accessing the nested Geocode object correctly
                # We check for both Geocode and geocode to be safe
                geo = est.get('Geocode') or est.get('geocode') or {}
                
                all_establishments.append({
                    'msoa_name_hint': est.get('LocalAuthorityName'), # Used for initial grouping
                    'business_name': est.get('BusinessName'),
                    'postcode': est.get('PostCode'),
                    'latitude': geo.get('latitude'),
                    'longitude': geo.get('longitude'),
                    'authority_id': auth_id
                })
            
            # Check if there are more pages
            meta = data.get('meta', {})
            if page >= meta.get('totalPages', 1):
                break
            page += 1
        
        time.sleep(0.2) # Compliance delay

    # --- Step 3: Clean and Display ---
    df = pd.DataFrame(all_establishments)
    
    # Convert lat/lon to numeric, errors='coerce' turns empty strings to NaN
    df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
    df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
    
    # Remove rows that still have no coordinates
    df_clean = df.dropna(subset=['latitude', 'longitude'])
    
    print(f"\nFinal count: {len(df_clean)} records with valid coordinates.")
    return df_clean

# Execute
df_london_takeaways = get_london_fast_food_data()

Fetching London Authority list...
Found 33 London boroughs.
Processing Barking and Dagenham (ID: 88)...
Processing Barnet (ID: 89)...
Processing Bexley (ID: 90)...
Processing Brent (ID: 91)...
Processing Bromley (ID: 92)...
Processing Camden (ID: 93)...
Processing City of London Corporation (ID: 95)...
Processing Croydon (ID: 94)...
Processing Ealing (ID: 96)...
Processing Enfield (ID: 97)...
Processing Greenwich (ID: 98)...
Processing Hackney (ID: 99)...
Processing Hammersmith and Fulham (ID: 100)...
Processing Haringey (ID: 101)...
Processing Harrow (ID: 102)...
Processing Havering (ID: 103)...
Processing Hillingdon (ID: 104)...
Processing Hounslow (ID: 105)...
Processing Islington (ID: 106)...
Processing Kensington and Chelsea (ID: 107)...
Processing Kingston-Upon-Thames (ID: 108)...
Processing Lambeth (ID: 109)...
Processing Lewisham (ID: 110)...
Processing Merton (ID: 111)...
Processing Newham (ID: 112)...
Processing Redbridge (ID: 113)...
Processing Richmond-Upon-Thames (ID: 114)

### Calculate takeaway density per MSOA

In [10]:
# Load the MSOA Geometries
MSOA_GEO = r"Middle_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V3_-4477917303172606123.geojson"
print("Loading MSOA boundaries...")
msoas_all = gpd.read_file((os.path.join(RAW_DIR, MSOA_GEO)))

# Filter boundaries to London only
msoas_london = msoas_all[msoas_all['MSOA21CD'].isin(london_msoas.msoa_code)].copy()
print(f"Filtered to {len(msoas_london)} London MSOAs.")

# Load takeaway data
print("Loading takeaway data...")
takeaways_gdf = gpd.GeoDataFrame(
    df_london_takeaways, 
    geometry=gpd.points_from_xy(df_london_takeaways.longitude, df_london_takeaways.latitude),
    crs="EPSG:4326"
)

# Project both to British National Grid (EPSG:27700)
# Essential for accurate area calculations and spatial joins
print("...projecting to British National Grid (EPSG:27700)...")
msoas_london = msoas_london.to_crs(epsg=27700)
takeaways_gdf = takeaways_gdf.to_crs(epsg=27700)

# Spatial Join: Count takeaways per MSOA
# This identifies which MSOA polygon each point is 'within'
print("...performing spatial join to count takeaways per MSOA...")
joined = gpd.sjoin(takeaways_gdf, msoas_london[['MSOA21CD', 'geometry']], predicate='within')
counts = joined.groupby('MSOA21CD').size().reset_index(name='takeaway_count')

# Merge the counts into the geometry dataframe
print("...merging takeaway counts into MSOA geometries...")
takeaway_calcs = msoas_london.merge(counts, on='MSOA21CD', how='left').fillna({'takeaway_count': 0})

# Normalise by area to get density (takeaways per km2)
print("...calculating takeaway density...")
takeaway_calcs['fast_food_density'] = takeaway_calcs['takeaway_count'] / (takeaway_calcs.geometry.area / 1000000)

df_fast_food = takeaway_calcs[['MSOA21CD', 'fast_food_density', 'BNG_E', 'BNG_N']].copy()
df_fast_food = df_fast_food.rename(columns={'MSOA21CD': 'msoa_code', 'BNG_E': 'BNG_east', 'BNG_N': 'BNG_north'})
print(f"... Fast food density calculated for {len(df_fast_food)} MSOAs.")

Loading MSOA boundaries...
Filtered to 1002 London MSOAs.
Loading takeaway data...
...projecting to British National Grid (EPSG:27700)...
...performing spatial join to count takeaways per MSOA...
...merging takeaway counts into MSOA geometries...
...calculating takeaway density...
... Fast food density calculated for 1002 MSOAs.


## Merge for the final dataset

In [11]:
# --- 5. Join variables into one table ---
print("Merging all data into master table...")

# Join all blocks onto the 983 London MSOAs using MSOA21CD
df_merge = pd.merge(df_obesity, df_green, on='msoa_code', how='left')
df_merge = pd.merge(df_merge, df_imd, on='msoa_code', how='left')
df_merge = pd.merge(df_merge, df_ptal, on='msoa_code', how='left')
df_merge = pd.merge(df_merge, df_fast_food, on='msoa_code', how='left')

print("...merged master table created")

Merging all data into master table...
...merged master table created


## Final Checks

### Scaling and NaNs

As the final variables are all in differing units we use *StandardScaler* from *sklearn* to ensure that values are of a common scale.

There are also NaNs values in the obesity and green spaces variables. As *child_obesity_rate* is our target variable we drop the NaN values as is standard practice. For the independent variable of *avg_size_green_1km_m2* we then use *KNNImputer* to complete the missing values based on the geographical nearest neighbours. 

In [12]:
# Drop rows where child_obesity_rate is missing
# These are NCMP-suppressed MSOAs with insufficient child sample sizes
df_merge = df_merge.dropna(subset=['child_obesity_rate']).reset_index(drop=True)
print(f'Retained {len(df_merge)} MSOAs after dropping suppressed obesity rates')

# Select all numerical columns needed for the model + coordinates for geography
# Note: we do not include the target variable (obesity) in the scaling process
cols_to_process = [
    'avg_size_green_1km_m2', 'employment_score', 'education_score', 'crime_score', 
    'idaci_score', 'ptal_accessibility_index', 'fast_food_density', 'BNG_east', 'BNG_north'
]

# Scaling before imputing so distances are on the same scale for KNN imputation
scaler = StandardScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df_merge[cols_to_process]), 
    columns=cols_to_process
)

# Impute on scaled values
imputer = KNNImputer(n_neighbors=5)
df_imputed_scaled = imputer.fit_transform(df_scaled)

# Inverse transform back to original scale
# Crucial to ensure the imputed values are on the same scale as the original data
df_imputed = scaler.inverse_transform(df_imputed_scaled)
df_merge[cols_to_process] = df_imputed

Retained 980 MSOAs after dropping suppressed obesity rates


## Export the final CSV

In [13]:
# Save Master File
output_path = os.path.join(PROCESSED_DIR, "london_obesity_master_dataset.csv")
df_merge.to_csv(output_path, index=False)

# Cleanup
if os.path.exists("temp_obesity.csv"): os.remove("temp_obesity.csv")

print(f"--- Process Complete ---")
print(f"Total London MSOAs: {len(df_merge)}")

--- Process Complete ---
Total London MSOAs: 980
